# COW-CLM-001 — Minimal Functional Fork

Frozen hosted-GPU orchestration for protocol v1.5 / seed `26090511`. Terminal scientific PASS and FAIL are both published. Required Kaggle Secrets: `HF_TOKEN`, `GITHUB_TOKEN`.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

BRANCH = 'codex/cow-clm-001-minimal-functional-fork'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'
ROOT = Path('/kaggle/working/mini-cells')
TRANSFORMERS_VERSION = '5.16.1'
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', f'transformers=={TRANSFORMERS_VERSION}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
installed_transformers = subprocess.check_output(
    [sys.executable, '-c', 'import transformers; print(transformers.__version__)'],
    text=True,
).strip()
if installed_transformers != TRANSFORMERS_VERSION:
    raise RuntimeError(
        f'hosted Transformers pin failed: expected {TRANSFORMERS_VERSION}, found {installed_transformers}'
    )
print(f'transformers_version={installed_transformers}')
print('checkout_ready=True')


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
if not os.environ['HF_TOKEN'] or not os.environ['GITHUB_TOKEN']:
    raise RuntimeError('HF_TOKEN and GITHUB_TOKEN Kaggle Secrets are required')
print('hf_token_loaded=True')
print('github_token_loaded=True')

subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_cow_clm.py', 'tests/test_cow_clm_001_protocol.py'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/research/cow_clm_001/publish.py', '--branch', BRANCH, '--preflight-only'], cwd=ROOT, check=True)
print('cpu_and_publish_preflight=PASS')


In [ ]:
protocol_path = ROOT / 'research/validations/cow-clm-001/protocol.json'
protocol = json.loads(protocol_path.read_text())
protocol_sha = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
decision_path = ROOT / 'artifacts/experiments/cow-clm-001/decision.json'
recovered = False
if decision_path.is_file():
    decision = json.loads(decision_path.read_text())
    recovered = (
        decision.get('terminal_result') in {'PASS', 'FAIL'}
        and decision.get('protocol_sha256') == protocol_sha
        and decision.get('implementation_git_blobs') == protocol.get('implementation_git_blobs')
        and int(decision.get('seed', -1)) == int(protocol['seed'])
    )
print(f'protocol_version={protocol["protocol_version"]}')
print(f'protocol_sha256={protocol_sha}')
print(f'matching_terminal_artifact={recovered}')


In [ ]:
if not recovered:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is required by the frozen COW-CLM-001 protocol')
    count = torch.cuda.device_count()
    print(f'cuda_device_count={count}')
    for index in range(count):
        print(f'cuda:{index}={torch.cuda.get_device_name(index)}')
    command = [sys.executable, 'scripts/research/cow_clm_001/run_frozen.py', '--device', 'cuda:0']
    if count >= 2:
        command += ['--capability-device', 'cuda:1', '--parallel-tracks']
        print('gpu_policy=parallel_independent_tracks')
    else:
        print('gpu_policy=sequential_single_gpu')
    subprocess.run(command, cwd=ROOT, check=True)
    subprocess.run([sys.executable, 'scripts/research/cow_clm_001/publish.py', '--seed', str(protocol['seed']), '--branch', BRANCH], cwd=ROOT, check=True)
else:
    print('gpu_run_skipped=matching_durable_terminal_artifact')


In [ ]:
if recovered:
    final = json.loads(decision_path.read_text())
else:
    summary_path = ROOT / 'results/cow-clm-001' / f'seed-{protocol["seed"]}' / 'seed_summary.json'
    final = json.loads(summary_path.read_text())
print(json.dumps(final, indent=2, sort_keys=True))
